In [4]:
import sys
sys.path.insert(0, "/home/nfm/ViT-Prisma/src")

from vit_prisma.sae import VisionModelSAERunnerConfig
from vit_prisma.sae import VisionSAETrainer
from vit_prisma.transforms import get_clip_val_transforms


import torchvision
import torch

from torch.utils.data import DataLoader, Subset


# Train the SAE

In [5]:

# Load an SAE
from huggingface_hub import hf_hub_download, list_repo_files
from vit_prisma.sae import SparseAutoencoder

def load_sae(repo_id, file_name, config_name):
    # Step 1: Download SAE weights and SAE config from Hugginface
    sae_path = hf_hub_download(repo_id, file_name) # Download SAE weights
    hf_hub_download(repo_id, config_name) # Download SAE config

    # Step 2: Now load the pretrained SAE weights from where you just downloaded them
    print(f"Loading SAE from {sae_path}...")
    sae = SparseAutoencoder.load_from_pretrained(sae_path) # This now automatically gets the config.json file in that folder and converts it into a VisionSAERunnerConfig object
    return sae

# Change the repo_id to the Huggingface repo of your chosen SAE. See /docs for a list of SAEs.
repo_id = "Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05" 

file_name = "weights.pt" # Usually weights.pt but may have slight naming variation. See the chosen HF repo for the exact file name
config_name = "config.json"

sae = load_sae(repo_id, file_name, config_name)

2026-04-19 00:46:00 DEBUG:urllib3.connectionpool: Resetting dropped connection: huggingface.co
2026-04-19 00:46:00 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/resolve/main/weights.pt HTTP/1.1" 302 0
2026-04-19 00:46:00 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/resolve/main/config.json HTTP/1.1" 307 0
2026-04-19 00:46:00 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/1e0dcc98c63aed3ce28f0387027cf8df6784fef1/config.json HTTP/1.1" 200 0


Loading SAE from /home/nfm/.cache/huggingface/hub/models--Prisma-Multimodal--sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/snapshots/1e0dcc98c63aed3ce28f0387027cf8df6784fef1/weights.pt...


2026-04-19 00:46:01 WARNING:root: Deprecated field 'total_training_images' found in config. It will be ignored.
2026-04-19 00:46:01 WARNING:root: Deprecated field 'total_training_tokens' found in config. It will be ignored.
2026-04-19 00:46:01 WARNING:root: Deprecated field 'd_sae' found in config. It will be ignored.
2026-04-19 00:46:01 INFO:root: n_tokens_per_buffer (millions): 0.032
2026-04-19 00:46:01 INFO:root: Lower bound: n_contexts_per_buffer (millions): 0.00064
2026-04-19 00:46:01 INFO:root: Total training steps: 158691
2026-04-19 00:46:01 INFO:root: Total training images: 13000000
2026-04-19 00:46:01 INFO:root: Total wandb updates: 1586
2026-04-19 00:46:01 INFO:root: Expansion factor: 64
2026-04-19 00:46:01 INFO:root: n_tokens_per_feature_sampling_window (millions): 204.8
2026-04-19 00:46:01 INFO:root: n_tokens_per_dead_feature_window (millions): 1024.0
2026-04-19 00:46:01 INFO:root: We will reset the sparsity calculation 158 times.
2026-04-19 00:46:01 INFO:root: Number token

In [23]:
from vit_prisma.models.model_loader import load_hooked_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = sae.cfg.model_name
# model_name = "vit_base_patch16_224"
model = load_hooked_model(model_name)
model.to(DEVICE)

2026-04-19 00:55:27 INFO:root: Model 'open-clip:laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K' is supported and passes tests.
2026-04-19 00:55:27 INFO:root: model_id download_pretrained_from_hf: laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K
2026-04-19 00:55:27 DEBUG:urllib3.connectionpool: Resetting dropped connection: huggingface.co


*****Loading model 'open-clip:laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K' of type 'VISION'


2026-04-19 00:55:27 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K/resolve/main/open_clip_config.json HTTP/1.1" 307 0
2026-04-19 00:55:27 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K/f0e2ffa09cbadab3db6a261ec1ec56407ce42912/open_clip_config.json HTTP/1.1" 200 0
2026-04-19 00:55:28 INFO:root: model_id download_pretrained_from_hf: laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K
2026-04-19 00:55:28 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K/resolve/main/open_clip_pytorch_model.bin HTTP/1.1" 302 0
2026-04-19 00:55:29 INFO:root: visual projection shape: torch.Size([768, 512])
2026-04-19 00:55:30 INFO:root: Loaded pretrained model open-clip:laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K into HookedTransformer


HookedViT(
  (embed): PatchEmbedding(
    (proj): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32))
  )
  (hook_embed): HookPoint()
  (pos_embed): PosEmbedding()
  (hook_pos_embed): HookPoint()
  (hook_full_embed): HookPoint()
  (ln_pre): LayerNorm(
    (hook_scale): HookPoint()
    (hook_normalized): HookPoint()
  )
  (hook_ln_pre): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): Ho

In [20]:
# Put your ImageNet Paths here
from vit_prisma.transforms import get_clip_val_transforms

imagenet_train_path = '/home/nfm/data_prisma/imagenet_val/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train'
imagenet_validation_path = '/home/nfm/data_prisma/imagenet_val/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val'

data_transforms = get_clip_val_transforms()
train_dataset = torchvision.datasets.ImageFolder(imagenet_train_path, transform=data_transforms)
eval_dataset = torchvision.datasets.ImageFolder(imagenet_validation_path, transform=data_transforms)


In [24]:
sae_trainer_cfg = VisionModelSAERunnerConfig( 
    hook_point_layer=10,
    layer_subtype='hook_resid_post',
    dataset_name="imagenet",
    feature_sampling_window=1000,
    activation_fn_str='relu',
    checkpoint_path = '/home/nfm/ViT-Prisma/demos/sae_ckpts'
)

2026-04-19 00:55:40 INFO:root: n_tokens_per_buffer (millions): 0.032
2026-04-19 00:55:40 INFO:root: Lower bound: n_contexts_per_buffer (millions): 0.00064
2026-04-19 00:55:40 INFO:root: Total training steps: 15869
2026-04-19 00:55:40 INFO:root: Total training images: 1300000
2026-04-19 00:55:40 INFO:root: Total wandb updates: 1586
2026-04-19 00:55:40 INFO:root: Expansion factor: 16
2026-04-19 00:55:40 INFO:root: n_tokens_per_feature_sampling_window (millions): 204.8
2026-04-19 00:55:40 INFO:root: n_tokens_per_dead_feature_window (millions): 1024.0
2026-04-19 00:55:40 INFO:root: We will reset the sparsity calculation 15 times.
2026-04-19 00:55:40 INFO:root: Number tokens in sparsity calculation window: 4.10e+06
2026-04-19 00:55:40 INFO:root: Gradient clipping with max_norm=1.0
2026-04-19 00:55:40 INFO:root: Using SAE initialization method: independent


In [27]:
sae_trainer_cfg.num_workers = 6

In [28]:
trainer = VisionSAETrainer(sae_trainer_cfg, model, train_dataset, eval_dataset)
sae = trainer.run()

2026-04-19 01:00:49 INFO:root: get_activation_fn received: activation_fn=relu, kwargs={}
2026-04-19 01:00:52 DEBUG:asyncio: Using selector: EpollSelector
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
2026-04-19 01:01:10 DEBUG:git.cmd: Popen(['git', 'version'], cwd=/home/nfm/ViT-Prisma/demos, stdin=None, shell=False, universal_newlines=False)
2026-04-19 01:01:10 DEBUG:git.cmd: Popen(['git', 'version'], cwd=/home/nfm/ViT-Prisma/demos, stdin=None, shell=False, universal_newlines=False)
2026-04-19 01:01:10 DEBUG:git.util: sys.platform='linux', git_executable='git'
2026-04-19 01:01:10 DEBUG:git.cmd: Popen(['git', 'cat-file', '--batch-check'], cwd=/home/nfm/ViT-Prisma, stdin=<valid stream>, shell=False, universal_newlines=False)


Training SAE:   0%|          | 0/65000000 [00:00<?, ?it/s]2026-04-19 01:01:19 DEBUG:PIL.TiffImagePlugin: tag: Make (271) - type: string (2) Tag Location: 22 - Data Location: 122 - value: b'Canon\x00'
2026-04-19 01:01:19 DEBUG:PIL.TiffImagePlugin: tag: Model (272) - type: string (2) Tag Location: 34 - Data Location: 128 - value: b'Canon EOS DIGITAL REBEL XT\x00'
2026-04-19 01:01:19 DEBUG:PIL.TiffImagePlugin: tag: Orientation (274) - type: short (3) - value: b'\x01\x00'
2026-04-19 01:01:19 DEBUG:PIL.TiffImagePlugin: tag: XResolution (282) - type: rational (5) Tag Location: 58 - Data Location: 155 - value: b'H\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:01:19 DEBUG:PIL.TiffImagePlugin: tag: YResolution (283) - type: rational (5) Tag Location: 70 - Data Location: 163 - value: b'H\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:01:19 DEBUG:PIL.TiffImagePlugin: tag: ResolutionUnit (296) - type: short (3) - value: b'\x02\x00'
2026-04-19 01:01:19 DEBUG:PIL.TiffImagePlugin: tag: DateTime (306) - type

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/ee158df6-tinyclip_sae_16_hyperparam_sweep_lr/n_images_130007.pt


Training SAE: Loss: 0.0171, MSE Loss: 0.0132, L1 Loss: 0.0039, L0: 10.8912:  14%|█▍        | 9068544/65000000 [04:40<28:56, 32207.09it/s]2026-04-19 01:05:52 DEBUG:PIL.TiffImagePlugin: tag: Make (271) - type: string (2) Tag Location: 22 - Data Location: 4688 - value: b'EASTMAN KODAK COMPANY\x00'
2026-04-19 01:05:52 DEBUG:PIL.TiffImagePlugin: tag: Model (272) - type: string (2) Tag Location: 34 - Data Location: 4710 - value: b'KODAK C310 DIGITAL CAMERA\x00'
2026-04-19 01:05:52 DEBUG:PIL.TiffImagePlugin: tag: XResolution (282) - type: rational (5) Tag Location: 46 - Data Location: 206 - value: b'\xe6\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:05:52 DEBUG:PIL.TiffImagePlugin: tag: YResolution (283) - type: rational (5) Tag Location: 58 - Data Location: 214 - value: b'\xe6\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:05:52 DEBUG:PIL.TiffImagePlugin: tag: ResolutionUnit (296) - type: short (3) - value: b'\x02\x00'
2026-04-19 01:05:52 DEBUG:PIL.TiffImagePlugin: tag: Software (305) - type: strin

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/ee158df6-tinyclip_sae_16_hyperparam_sweep_lr/n_images_260014.pt


...

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/ee158df6-tinyclip_sae_16_hyperparam_sweep_lr/n_images_1040056.pt


Training SAE: Loss: 0.0158, MSE Loss: 0.0116, L1 Loss: 0.0042, L0: 12.2909:  84%|████████▍ | 54718464/65000000 [28:54<05:21, 32029.49it/s]2026-04-19 01:30:19 DEBUG:PIL.TiffImagePlugin: tag: Make (271) - type: string (2) Tag Location: 22 - Data Location: 4688 - value: b'EASTMAN KODAK COMPANY\x00'
2026-04-19 01:30:19 DEBUG:PIL.TiffImagePlugin: tag: Model (272) - type: string (2) Tag Location: 34 - Data Location: 4710 - value: b'KODAK C310 DIGITAL CAMERA\x00'
2026-04-19 01:30:19 DEBUG:PIL.TiffImagePlugin: tag: XResolution (282) - type: rational (5) Tag Location: 46 - Data Location: 206 - value: b'\xe6\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:30:19 DEBUG:PIL.TiffImagePlugin: tag: YResolution (283) - type: rational (5) Tag Location: 58 - Data Location: 214 - value: b'\xe6\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:30:19 DEBUG:PIL.TiffImagePlugin: tag: ResolutionUnit (296) - type: short (3) - value: b'\x02\x00'
2026-04-19 01:30:19 DEBUG:PIL.TiffImagePlugin: tag: Software (305) - type: stri

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/ee158df6-tinyclip_sae_16_hyperparam_sweep_lr/n_images_1170063.pt


Training SAE: Loss: 0.0159, MSE Loss: 0.0117, L1 Loss: 0.0042, L0: 12.3610:  91%|█████████ | 59027456/65000000 [31:11<03:08, 31637.87it/s]2026-04-19 01:32:26 DEBUG:PIL.TiffImagePlugin: tag: Make (271) - type: string (2) Tag Location: 22 - Data Location: 122 - value: b'Canon\x00'
2026-04-19 01:32:26 DEBUG:PIL.TiffImagePlugin: tag: Model (272) - type: string (2) Tag Location: 34 - Data Location: 128 - value: b'Canon EOS DIGITAL REBEL XT\x00'
2026-04-19 01:32:26 DEBUG:PIL.TiffImagePlugin: tag: Orientation (274) - type: short (3) - value: b'\x01\x00'
2026-04-19 01:32:26 DEBUG:PIL.TiffImagePlugin: tag: XResolution (282) - type: rational (5) Tag Location: 58 - Data Location: 155 - value: b'H\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:32:26 DEBUG:PIL.TiffImagePlugin: tag: YResolution (283) - type: rational (5) Tag Location: 70 - Data Location: 163 - value: b'H\x00\x00\x00\x01\x00\x00\x00'
2026-04-19 01:32:26 DEBUG:PIL.TiffImagePlugin: tag: ResolutionUnit (296) - type: short (3) - value: b'\x0

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/ee158df6-tinyclip_sae_16_hyperparam_sweep_lr/n_images_1300070.pt
